In [ ]:
import sys
sys.path.insert(0, '..')
from support.paths import resolve
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from joblib import Parallel, delayed

In [9]:
%run ../0_Config/0_variables.ipynb

In [ ]:
# Model constants
_SPIKE_UPWEIGHT       = 10.0
_DIP_UPWEIGHT         = 7.0
_MIN_SPIKE_TRAIN      = 20
EARLY_STOPPING_ROUNDS = 75
SPIKE_ES_ROUNDS       = 50

# LightGBM hyperparameters
from parameters import (
    LGBM_BASE_PARAMS,
    LGBM_BASE_L2_PARAMS,
    LGBM_CLF_PARAMS,
    LGBM_SPIKE_PARAMS,
    LGBM_SPIKE_Q_PARAMS,
    LGBM_DIP_CLF_PARAMS,
    LGBM_DIP_PARAMS,
)

In [10]:
X_train = pd.read_parquet("Data/X_train.parquet")
X_validate = pd.read_parquet("Data/X_validate.parquet")
X_test = pd.read_parquet("Data/X_test.parquet")
y_train = pd.read_parquet("Data/y_train.parquet")
y_validate = pd.read_parquet("Data/y_validate.parquet")
y_test = pd.read_parquet("Data/y_test.parquet")

In [ ]:
def train_seq2seq(X_train, X_validate, y_train, y_validate):

    n_horizons   = int(os.environ["HORIZON_HOURS"]) * (60 // int(os.environ["OUTPUT_RESOLUTION"]))
    horizon_list = list(range(1, n_horizons + 1))

    # 1D sample-weight array of shape (n_train_rows,) — not a feature, never enters X_train.
    # Each entry scales how much that row contributes to the loss during LightGBM training.
    # linspace(0, 1.5, n) spaces exponents evenly across rows (oldest → newest), so
    # exp() produces weights rising from 1.0 (oldest row) to ~4.5 (newest row),
    # assuming X_train is ordered chronologically. Computed once and shared across all horizons.
    _chronological_weights = np.exp(np.linspace(0.0, 1.5, len(X_train))).astype(np.float32)

   

    # Data processing helper
    def _create_y_derivatives_per_target_col(target_col):
        """Compute all y (target) transformations for a given horizon."""
        PRICE_TRANSFORM_SCALE = 100
        STANDARD_CLIP_PERCENTILE = 97.0
        SPIKE_THRESHOLD = 150.0
        NEGATIVE_PRICE_THRESHOLD = 0.0

        # y_train/y_validate hold all horizons as columns; slice the one being fitted
        # and convert to float32 to match LightGBM's expected dtype
        y_train_individual_target = y_train[target_col].values.astype(np.float32)
        y_validate_individual_target = y_validate[target_col].values.astype(np.float32)

        # Compute sample weights for standard model training
        # Returns a 1D sample-weight array of shape (n_train_rows,) combining two effects:
        #   - recency: weights already rise from 1.0 → ~4.5 via _chronological_weights
        #   - spike rows (price > threshold) are further multiplied by 3x
        # Net result: a recent spike row is weighted up to ~13.4x vs an old normal row,
        # preventing the standard model's loss from treating spikes as negligible noise.
        y_train_loss_weights = (_chronological_weights * np.where(y_train_individual_target > SPIKE_THRESHOLD, 3.0, 1.0)).astype(np.float32)

        # Compute clip threshold from training data
        y_train_individual_clipped = float(np.percentile(y_train_individual_target, STANDARD_CLIP_PERCENTILE))

        # arcsinh(y / 100) is a variance-stabilising transform that compresses large values
        # while preserving sign — unlike log, it handles negative prices. np.minimum clips
        # at the 97th-percentile threshold before transforming so extreme spikes don't
        # dominate the standard model's loss; the spike models handle those separately.
        y_train_individual_clipped_transformed = np.arcsinh(np.minimum(y_train_individual_target, y_train_individual_clipped) / PRICE_TRANSFORM_SCALE).astype(np.float32)
        y_validation_individual_clipped_transformed = np.arcsinh(np.minimum(y_validate_individual_target, y_train_individual_clipped) / PRICE_TRANSFORM_SCALE).astype(np.float32)

        # Same arcsinh transform but on the uncapped target — full price range including
        # extreme spikes and negative prices, which the specialist models are trained on.
        y_train_individual_unclipped_transformed = np.arcsinh(y_train_individual_target / PRICE_TRANSFORM_SCALE).astype(np.float32)
        y_validation_individual_unclipped_transformed = np.arcsinh(y_validate_individual_target / PRICE_TRANSFORM_SCALE).astype(np.float32)

        # Compute positive spike labels
        spike_labels_train = (y_train_individual_target > SPIKE_THRESHOLD).astype(np.float32)
        spike_labels_validate = (y_validate_individual_target > SPIKE_THRESHOLD).astype(np.float32)

        # Compute negative spike labels
        negative_labels_train = (y_train_individual_target < NEGATIVE_PRICE_THRESHOLD).astype(np.float32)
        negative_labels_validate = (y_validate_individual_target < NEGATIVE_PRICE_THRESHOLD).astype(np.float32)

        return {
            # 1 - target clipped and transformed 
            'y_train_individual_clipped_transformed': y_train_individual_clipped_transformed,
            'y_validation_individual_clipped_transformed': y_validation_individual_clipped_transformed,

            # 2 - target no clipped but transformed
            'y_train_individual_unclipped_transformed': y_train_individual_unclipped_transformed,
            'y_validation_individual_unclipped_transformed': y_validation_individual_unclipped_transformed,
           
            # 3 - Postive target spike labels
            'spike_labels_train': spike_labels_train,
            'spike_labels_validate': spike_labels_validate,

            # 4 - Negative target spike labels
            'negative_labels_train': negative_labels_train,
            'negative_labels_validate': negative_labels_validate,

            # 5 - Weighting curve of target inc recency and spikes 
            'y_train_loss_weights': y_train_loss_weights,
        }


    def _fit_one(horizon: int):
        target_col = f"h{horizon}"
        targets = _create_y_derivatives_per_target_col(target_col)

        # Extract pre-computed data
        y_train_individual_clipped_transformed = targets['y_train_individual_clipped_transformed']
        y_validation_individual_clipped_transformed = targets['y_validation_individual_clipped_transformed']
        y_train_individual_unclipped_transformed = targets['y_train_individual_unclipped_transformed']
        y_validation_individual_unclipped_transformed = targets['y_validation_individual_unclipped_transformed']
        y_train_loss_weights = targets['y_train_loss_weights']
        spike_labels_train = targets['spike_labels_train']
        spike_labels_validate = targets['spike_labels_validate']
        negative_labels_train = targets['negative_labels_train']
        negative_labels_validate = targets['negative_labels_validate']

        n_spikes_train = int(spike_labels_train.sum())
        n_negative_train = int(negative_labels_train.sum())

        spike_clf = None
        spike_reg = None
        spike_qreg = None
        negative_clf = None
        negative_reg = None

        # 1a. Standard L1 model (MAE-optimal, clipped target)
        standard_l1_m = lgb.LGBMRegressor(**LGBM_BASE_PARAMS)
        standard_l1_m.fit(
            X_train, y_train_individual_clipped_transformed,
            sample_weight=y_train_loss_weights,
            eval_set=[(X_validate, y_validation_individual_clipped_transformed)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )

        # 1b. Standard L2 model (MSE-optimal, uncapped target)
        standard_l2_m = lgb.LGBMRegressor(**LGBM_BASE_L2_PARAMS)
        standard_l2_m.fit(
            X_train, y_train_individual_unclipped_transformed,
            sample_weight=y_train_loss_weights,
            eval_set=[(X_validate, y_validation_individual_unclipped_transformed)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )

        # 2. Spike classifier
        if n_spikes_train >= _MIN_SPIKE_TRAIN:
            clf = lgb.LGBMClassifier(**LGBM_CLF_PARAMS)
            clf.fit(
                X_train, spike_labels_train,
                sample_weight=_chronological_weights,
                eval_set=[(X_validate, spike_labels_validate)],
                callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
            )
            spike_clf = clf

            # 3. Spike regressor (full range, upweighted spike rows)
            spike_weights = _chronological_weights * np.where(spike_labels_train > 0, _SPIKE_UPWEIGHT, 1.0)

            sreg = lgb.LGBMRegressor(**LGBM_SPIKE_PARAMS)
            sreg.fit(
                X_train, y_train_individual_unclipped_transformed,
                sample_weight=spike_weights,
                eval_set=[(X_validate, y_validation_individual_unclipped_transformed)],
                callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
            )
            spike_reg = sreg

            qreg = lgb.LGBMRegressor(**LGBM_SPIKE_Q_PARAMS)
            qreg.fit(
                X_train, y_train_individual_unclipped_transformed,
                sample_weight=spike_weights,
                eval_set=[(X_validate, y_validation_individual_unclipped_transformed)],
                callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
            )
            spike_qreg = qreg

            # 4. Negative price classifier + regressor (prices below zero)
            if n_negative_train >= _MIN_SPIKE_TRAIN:
                negative_clf_m = lgb.LGBMClassifier(**LGBM_DIP_CLF_PARAMS)
                negative_clf_m.fit(
                    X_train, negative_labels_train,
                    sample_weight=_chronological_weights,
                    eval_set=[(X_validate, negative_labels_validate)],
                    callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
                )
                negative_clf = negative_clf_m

                negative_weights = _chronological_weights * np.where(negative_labels_train > 0, _DIP_UPWEIGHT, 1.0)
                negative_reg_m = lgb.LGBMRegressor(**LGBM_DIP_PARAMS)
                negative_reg_m.fit(
                    X_train, y_train_individual_unclipped_transformed,
                    sample_weight=negative_weights,
                    eval_set=[(X_validate, y_validation_individual_unclipped_transformed)],
                    callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
                )
                negative_reg = negative_reg_m

        return standard_l1_m, standard_l2_m, spike_clf, spike_reg, spike_qreg, negative_clf, negative_reg

    trained_models = Parallel(n_jobs=-1, prefer="threads")(
        delayed(_fit_one)(h) for h in horizon_list
    )

    standard_l1_models, standard_l2_models, spike_clfs, spike_regs, spike_qregs, negative_clfs, negative_regs = map(list, zip(*trained_models))
    return standard_l1_models, standard_l2_models, spike_clfs, spike_regs, spike_qregs, negative_clfs, negative_regs

In [ ]:
standard_l1_models, standard_l2_models, spike_clfs, spike_regs, spike_qregs, negative_clfs, negative_regs = train_seq2seq(X_train, X_validate, y_train, y_validate)